# Pipeline SF3D di Kaggle — dataset artefak budaya

Jalankan tahap A–E di Kaggle **GPU T4**. Input = Kaggle Dataset, output ke `/kaggle/working`.

**Sebelum mulai (panel kanan Kaggle):**
1. **Settings → Accelerator → GPU T4 x1**.
2. **Settings → Internet → ON** (perlu untuk pip + git + Hugging Face).
3. **Add Data →** cari & attach dataset foto artefak kamu.
4. **Add-ons → Secrets →** tambah secret `HF_TOKEN` (token Hugging Face, sudah accept license SF3D).

Sel 1 auto-deteksi folder foto (layout apa pun: flat `*.jpg` atau bernested). Tak perlu hardcode path.

In [ ]:
#@title 1. Cek GPU + temukan path dataset otomatis (robust ke layout apa pun)
import glob, os
from collections import Counter
# cegah fragmentasi VRAM T4 (OOM "reserved but unallocated"). HARUS sebelum torch
# init CUDA (sel 4), jadi diset paling awal di sel 1.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
!nvidia-smi -L
print('Dataset ter-attach:', glob.glob('/kaggle/input/*'))
# kumpulkan semua gambar di bawah /kaggle/input, pilih folder dgn gambar terbanyak
imgs = []
for ext in ('jpg', 'jpeg', 'png', 'webp'):
    imgs += glob.glob(f'/kaggle/input/**/*.{ext}', recursive=True)
    imgs += glob.glob(f'/kaggle/input/**/*.{ext.upper()}', recursive=True)
counts = Counter(os.path.dirname(p) for p in imgs)
INPUT = counts.most_common(1)[0][0] if counts else ''
print(f'total gambar terdeteksi: {len(imgs)}')
print('INPUT =', INPUT, '| jumlah di folder ini:', counts.get(INPUT, 0))
print('contoh:', [os.path.basename(p) for p in imgs[:3]] if imgs else 'KOSONG - attach dataset dulu')

In [ ]:
#@title 2. Pasang SF3D (internet harus ON) — satu resolve koheren, tetap numpy 2.x
%cd /kaggle/working
import os, importlib
if not os.path.exists('sf3d_repo'):
    !git clone -q https://github.com/Stability-AI/stable-fast-3d.git sf3d_repo

# ekstensi C++ SF3D
!pip install ./sf3d_repo/texture_baker ./sf3d_repo/uv_unwrapper

# SATU pip resolve koheren (hindari perang numpy yg bikin AttributeError _no_nep50_warning).
#  - transformers 4.43.x: dukung numpy>=2 (4.42 paksa <2 & bentrok seisi Kaggle)
#    TAPI masih punya find_pruneable_heads_and_indices (baru dihapus di transformers 5.0).
#  - gpytoolbox butuh numpy<2.3 & scipy<=1.15  -> kunci numpy 2.2.x.
#  - rembg<2.0.70: versi >=2.0.76 paksa numpy>=2.3 (bentrok gpytoolbox).
!pip install "transformers==4.43.4" jaxtyping pynanoinstantmeshes omegaconf trimesh einops safetensors huggingface_hub onnxruntime "rembg<2.0.70" gpytoolbox "numpy<2.3" "scipy<=1.15"

# verifikasi semua impor kunci (proses harus BERSIH: Restart kernel sebelum run sel ini)
for m in ('jaxtyping', 'gpytoolbox', 'pynanoinstantmeshes', 'omegaconf', 'trimesh', 'onnxruntime', 'rembg', 'einops', 'transformers'):
    importlib.import_module(m)
import transformers, numpy
print('transformers', transformers.__version__, '(4.43.x) | numpy', numpy.__version__, '(2.2.x)')
print('SEMUA DEP OK')

In [ ]:
#@title 3. Clone/UPDATE repo tim (paksa src terbaru) + login Hugging Face
REPO_URL = "https://github.com/eycoo/FP_GRAFKOM.git"  #@param {type:"string"}
import os, sys
if os.path.exists('/kaggle/working/FP_GRAFKOM'):
    # hard reset -> SELALU samakan dgn origin/master (hindari src basi penyebab patch tak jalan)
    !cd /kaggle/working/FP_GRAFKOM && git fetch -q origin && git reset --hard -q origin/master
else:
    !git clone -q $REPO_URL /kaggle/working/FP_GRAFKOM
print('commit src:')
!cd /kaggle/working/FP_GRAFKOM && git log -1 --oneline
sys.path.insert(0, '/kaggle/working/sf3d_repo')
sys.path.insert(0, '/kaggle/working/FP_GRAFKOM/pipeline/src')

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret('HF_TOKEN'))
print('HF login OK')

In [ ]:
#@title 4. Muat SF3D + config (reload paksa src terbaru, output ke /kaggle/working/outputs)
import importlib, torch
import config, inference, preprocess
importlib.reload(config); importlib.reload(inference); importlib.reload(preprocess)  # buang modul basi (anti-cache)
from config import PipelineCfg
from inference import SF3DReconstructor
from preprocess import load_rembg_session

cfg = PipelineCfg.load('/kaggle/working/FP_GRAFKOM/pipeline/configs/default.yaml',
                       **{'export.out_dir': '/kaggle/working/outputs'})

# ANTI-OOM utama T4: ekstraksi mesh SF3D bikin fitur (N_tet, 3*Cp) ~5GB sekaligus.
# query_chunk memecahnya per-chunk titik -> puncak VRAM turun drastis, mesh identik.
# MASIH OOM? kecilkan: 32768 / 16384. 0 = perilaku asli SF3D (akan OOM).
CHUNK = 65536   #@param {type:"integer"}
cfg.model.query_chunk = CHUNK

reconstructor = SF3DReconstructor(cfg.model)
m = reconstructor.model
patched = getattr(type(m), '_chunked_query', None)
print('cond_image_size:', getattr(m.cfg, 'cond_image_size', None),
      '| isosurface_resolution:', getattr(m.cfg, 'isosurface_resolution', None))
print('PATCH chunk aktif:', patched, '(harus =', CHUNK, ', kalau None patch TAK kepasang)')

rembg_session = load_rembg_session()
print('device:', reconstructor.device, '| dtype bobot:', next(m.parameters()).dtype)
print('VRAM model resident:', round(torch.cuda.memory_allocated()/1024**3, 2), 'GB')

In [ ]:
#@title 5. Inferensi batch (TIER atur resolusi & VRAM; LIMIT untuk uji cepat)
from run_pipeline import collect_images, process_one
from export_glb import write_manifest
from pathlib import Path
import torch, gc

LIMIT = 5     #@param {type:"integer"}  # 0 = semua foto
TIER  = "low"  #@param ["high", "mid", "low"]

# TIER -> knob nyata SF3D (sebelum ini TIER cuma suffix nama file, tak ubah VRAM).
# bake_resolution = beban VRAM terbesar di T4. Mulai 'low' biar tak OOM, naik bertahap.
TIER_PRESETS = {
    "high": dict(texture_resolution=1024, remesh="none",     vertex_count=-1),
    "mid":  dict(texture_resolution=512,  remesh="triangle", vertex_count=20000),
    "low":  dict(texture_resolution=256,  remesh="triangle", vertex_count=10000),
}
for k, v in TIER_PRESETS[TIER].items():
    setattr(cfg.reconstruct, k, v)
print('TIER', TIER, '->', TIER_PRESETS[TIER])

manifest = Path(cfg.export.out_dir) / 'manifest.jsonl'
images = collect_images(Path(INPUT))
if LIMIT: images = images[:LIMIT]
print(f'proses {len(images)} foto')
for i, img in enumerate(images, 1):
    gc.collect(); torch.cuda.empty_cache()  # bersihkan VRAM antar foto (cegah akumulasi)
    try:
        rec = process_one(img, reconstructor, cfg, rembg_session, TIER)
        write_manifest(rec, manifest)
        print(f"[{i}/{len(images)}] {img.name[:40]} -> {rec['seconds']}s, "
              f"VRAM {rec['peak_vram_gb']}GB, {rec['glb_bytes']//1024}KB")
    except Exception as e:
        torch.cuda.empty_cache()  # lepas VRAM yg ke-pin saat OOM, biar foto berikut lanjut
        print(f'[{i}] GAGAL {img.name[:40]}: {e}')

In [ ]:
#@title 6. Zip hasil untuk diunduh (Output panel Kaggle)
import shutil
shutil.make_archive('/kaggle/working/glb_outputs', 'zip', '/kaggle/working/outputs')
print('Unduh: /kaggle/working/glb_outputs.zip (tab Output)')
!ls -la /kaggle/working/outputs/glb | head

## Catatan Kaggle
- Sesi GPU bisa putus → set `LIMIT` kecil dulu, naikkan bertahap. `outputs/` + `manifest.jsonl` jadi titik serah-terima.
- Untuk **commit hasil** (versi reproducible): Save Version → output tersimpan, bisa dijadikan dataset turunan untuk Jobdesk 3 & 4.
- Jika `INPUT` salah, cek sel 1 — slug dataset menentukan path `/kaggle/input/<slug>/`.